##**Installing the package**

In [1]:
!pip install hypothesis==6.156.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.6 MB/s eta 0:00:00


##**Workspace Property-Based Test**

In [5]:
import math
from hypothesis import given, strategies as st, assume, note

# --- 1. HARM STAGE 2 WORKSPACE FILTER ---
def harm_stage2_workspace_check(x: float, y: float, z: float) -> tuple[float, float, float]:
    """
    HARM Stage 2 Workspace Check: Ensures coordinates are within a 1.0m reach radius.
    If a coordinate is out of bounds, it clips it to the nearest boundary point.
    """
    MAX_RADIUS = 1.0

    current_distance = math.sqrt(x**2 + y**2 + z**2)

    if current_distance <= MAX_RADIUS:
        return (x, y, z)

    scale_factor = MAX_RADIUS / current_distance
    clipped_x = x * scale_factor
    clipped_y = y * scale_factor
    clipped_z = max(-0.2, min(z * scale_factor, 0.8))

    return (clipped_x, clipped_y, clipped_z)


# --- 2. HYPOTHESIS PROPERTY-BASED TEST ---
@given(
    x=st.floats(min_value=-10.0, max_value=10.0),
    y=st.floats(min_value=-10.0, max_value=10.0),
    z=st.floats(min_value=-10.0, max_value=10.0)
)
def test_harm_workspace_property(x: float, y: float, z: float):
    # Skip invalid mathematical errors like NaN or Infinity
    assume(not math.isnan(x) and not math.isinf(x))
    assume(not math.isnan(y) and not math.isinf(y))
    assume(not math.isnan(z) and not math.isinf(z))

    # Skip origin point (0,0,0) to prevent division-by-zero math errors
    assume(x != 0.0 or y != 0.0 or z != 0.0)

    # Run the coordinate data through the HARM filter
    out_x, out_y, out_z = harm_stage2_workspace_check(x, y, z)
    final_distance = math.sqrt(out_x**2 + out_y**2 + out_z**2)

    # --- FIXED STRING FORMATTING HERE ---
    note(f"Testing input: ({x:.4f}, {y:.4f}, {z:.4f})")
    note(f"HARM Output: ({out_x:.4f}, {out_y:.4f}, {out_z:.4f}) -> Final Distance: {final_distance:.4f}m")

    # THE CORE PROPERTY
    assert final_distance <= 1.00001


# --- 3. TEST RUNNER ---
if __name__ == "__main__":
    print("🚀 Starting Hypothesis Property-Based Fuzzing Test (100 runs)...")
    try:
        # Explicitly telling hypothesis to run the test function
        test_harm_workspace_property()
        print("✅ SUCCESS: HARM successfully kept all 100 random coordinate attacks within safe boundaries!")
    except Exception as e:
        print("❌ TEST FAILED: Hypothesis discovered an issue:")
        print(e)

🚀 Starting Hypothesis Property-Based Fuzzing Test (100 runs)...
✅ SUCCESS: HARM successfully kept all 100 random coordinate attacks within safe boundaries!
